# DataCite, Crossref Metadata and Make Data Count Relations

Looks at how many connections each relation type has in DataCite and Crossref Metadata, and how many connections per source in Make Data Count.

| Dataset | Version |
|---------|---------|
| DataCite | 2026-02 Snasphot |
| Crossref Metadata | 2025-04-01 Snapshot |
| Data Citation Corpus |  2025-08-15 Snapshot |

In [9]:
%reload_ext sql
%sql duckdb:///../data/duckdb/db.db
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

## Crossref Metadata

### Max Out Degree By Relation Type
For one specific work_doi, how many different related_dois with a specific relation_type does it point out to?

In [10]:
%%sql
    
SELECT
  relation_type,
  MAX(out_degree) AS max_out_degree
FROM 
  relations.crossref_metadata_degrees
GROUP BY 
  relation_type
ORDER BY 
  max_out_degree DESC;

Running query in 'duckdb:///../data/duckdb/db.db'

relation_type,max_out_degree
has-part,9165
is-supplemented-by,518
has-preprint,372
has-related-material,341
references,269
is-comment-on,156
has-review,151
is-part-of,100
has-version,95
is-version-of,95


### Max In Degree by Relation Type
For one specific related_doi, how many different work_dois with a specific relation_type are pointing in at it?

In [11]:
%%sql

SELECT
  relation_type,
  MAX(in_degree) AS max_in_degree
FROM 
  relations.crossref_metadata_degrees
GROUP BY 
  relation_type
ORDER BY 
  max_in_degree DESC;

Running query in 'duckdb:///../data/duckdb/db.db'

relation_type,max_in_degree
is-part-of,9165
is-supplement-to,518
is-preprint-of,372
is-related-material,353
is-referenced-by,269
is-review-of,170
has-preprint,108
has-related-material,102
has-version,95
is-version-of,95


## DataCite

### Max Out Degree By Relation Type
For one specific work_doi, how many different related_dois with a specific relation_type does it point out to?

In [12]:
%%sql

SELECT
  relation_type,
  MAX(out_degree) AS max_out_degree
FROM 
  relations.datacite_degrees
GROUP BY 
  relation_type
ORDER BY 
  max_out_degree DESC;


Running query in 'duckdb:///../data/duckdb/db.db'

relation_type,max_out_degree
IsDerivedFrom,113792
HasPart,24848
IsCitedBy,6612
Cites,3693
HasVersion,2710
IsDocumentedBy,2398
References,1031
IsCompiledBy,1013
IsPartOf,779
IsMetadataFor,751


### Max In Degree by Relation Type
For one specific related_doi, how many different work_dois with a specific relation_type are pointing in at it?

In [13]:
%%sql

SELECT
  relation_type,
  MAX(in_degree) AS max_in_degree
FROM 
  relations.datacite_degrees
GROUP BY 
  relation_type
ORDER BY 
  max_in_degree DESC;

Running query in 'duckdb:///../data/duckdb/db.db'

relation_type,max_in_degree
References,4625319
IsDerivedFrom,2238627
IsDocumentedBy,1603491
IsCompiledBy,49196
IsPartOf,34833
HasMetadata,32480
IsNewVersionOf,21534
Cites,16566
IsSourceOf,15060
IsCitedBy,9446


## Make Data Count

### Max Out Degree By Source
For one specific work_doi, how many different dataset_dois does it point out to?

In [14]:
%%sql

WITH relations_with_dois AS (
  SELECT
    NULLIF(LOWER(TRIM(REGEXP_EXTRACT(r.publication, '10\.[0-9.]+/[^\s]+'))), '') AS work_doi,
    NULLIF(LOWER(TRIM(REGEXP_EXTRACT(r.dataset, '10\.[0-9.]+/[^\s]+'))), '') AS dataset_doi,
    r.source
  FROM 
    data_citation_corpus.relations r
),
unique_pairs AS (
  SELECT DISTINCT
    work_doi,
    dataset_doi,
    source
  FROM 
    relations_with_dois
  WHERE 
    work_doi IS NOT NULL 
    AND dataset_doi IS NOT NULL 
    AND work_doi <> dataset_doi
),
pair_degrees AS (
  SELECT
    source,
    COUNT(*) OVER (PARTITION BY work_doi, source) AS out_degree
  FROM 
    unique_pairs
)
SELECT
  source,
  MAX(out_degree) AS max_out_degree
FROM 
  pair_degrees
GROUP BY 
  source
ORDER BY 
  max_out_degree DESC;

Running query in 'duckdb:///../data/duckdb/db.db'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

source,max_out_degree
datacite,8387
eupmc,250
czi,122
asap,2


### Max In Degree by Source
For one specific dataset_doi, how many different work_dois are pointing in at it?

In [15]:
%%sql

WITH relations_with_dois AS (
  SELECT
    NULLIF(LOWER(TRIM(REGEXP_EXTRACT(r.publication, '10\.[0-9.]+/[^\s]+'))), '') AS work_doi,
    NULLIF(LOWER(TRIM(REGEXP_EXTRACT(r.dataset, '10\.[0-9.]+/[^\s]+'))), '') AS dataset_doi,
    r.source
  FROM 
    data_citation_corpus.relations r
),
unique_pairs AS (
  SELECT DISTINCT
    work_doi,
    dataset_doi,
    source
  FROM 
    relations_with_dois
  WHERE 
    work_doi IS NOT NULL 
    AND dataset_doi IS NOT NULL 
    AND work_doi <> dataset_doi
),
pair_degrees AS (
  SELECT
    source,
    COUNT(*) OVER (PARTITION BY dataset_doi, source) AS in_degree
  FROM 
    unique_pairs
)
SELECT
  source,
  MAX(in_degree) AS max_in_degree
FROM 
  pair_degrees
GROUP BY 
  source
ORDER BY 
  max_in_degree DESC;

Running query in 'duckdb:///../data/duckdb/db.db'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

source,max_in_degree
datacite,6189
eupmc,201
czi,2
asap,2
